# Goal of This Notebook

The objective of this notebook is to extract contextualized word representations from BERT and understand how they evolve across layers.

The paper *How Contextual are Contextualized Word Representations?* investigates how much a word's representation changes depending on its context. Before reproducing the paper's experiments, we first need to obtain the hidden representations generated by BERT.

The workflow is:

1. Load a pretrained BERT model and tokenizer.
2. Tokenize a sentence and inspect how BERT processes text.
3. Run a forward pass through the model.
4. Extract hidden states from every layer.
5. Locate a target word within the tokenized sequence.
6. Retrieve the word's representation at each layer.
7. Compare the representations of the same word across different contexts.

The output of this notebook is a set of contextual embeddings for a target word. These embeddings will be used in the next phase to compute self-similarity scores and measure how contextualized BERT representations become across layers.

CELL 1 -Imports

In [2]:
import torch
import numpy as np

from transformers import BertTokenizer, BertModel

CELL 2 -Load Model

In [3]:
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

model = BertModel.from_pretrained(
    "bert-base-uncased",
    output_hidden_states=True
)

model.eval()

print("Model loaded!")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

c:\Users\ac tech\Desktop\modenLLMSin2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ac tech\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!


CELL 3 - Test Sentence

Eventually we'll use hundreds or thousands of sentences.

For now one sentence is enough to understand the pipeline.

In [4]:
sentence = "The bank approved the loan."

CELL 4 - Inspect Tokenization

In [5]:
tokens = tokenizer.tokenize(sentence)

print(tokens)

['the', 'bank', 'approved', 'the', 'loan', '.']


CELL 5 -tokenize for BERT

BERT cannot read text directly.
It only understands numbers.
The tokenizer converts
into tokens and IDs.

In [6]:
inputs = tokenizer(
    sentence,
    return_tensors="pt"
)

print(inputs)

{'input_ids': tensor([[ 101, 1996, 2924, 4844, 1996, 5414, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


Convert text into tensors.

CELL 6 - Run forward pass

The sentence flows through all transformer layers.
Each layer creates a new representation.

In [7]:
with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.hidden_states

CELL 7 - Understand the layers

In [8]:
print("Number of layers:", len(hidden_states))

Number of layers: 13


CELL 8 - Inspect Shapes

In [9]:
for i, layer in enumerate(hidden_states):
    print(f"Layer {i}: {layer.shape}")

Layer 0: torch.Size([1, 8, 768])
Layer 1: torch.Size([1, 8, 768])
Layer 2: torch.Size([1, 8, 768])
Layer 3: torch.Size([1, 8, 768])
Layer 4: torch.Size([1, 8, 768])
Layer 5: torch.Size([1, 8, 768])
Layer 6: torch.Size([1, 8, 768])
Layer 7: torch.Size([1, 8, 768])
Layer 8: torch.Size([1, 8, 768])
Layer 9: torch.Size([1, 8, 768])
Layer 10: torch.Size([1, 8, 768])
Layer 11: torch.Size([1, 8, 768])
Layer 12: torch.Size([1, 8, 768])


CELL 9 -See special Tokens

In [10]:
input_ids = inputs["input_ids"][0]

tokens_with_special = tokenizer.convert_ids_to_tokens(
    input_ids
)

print(tokens_with_special)

['[CLS]', 'the', 'bank', 'approved', 'the', 'loan', '.', '[SEP]']


CELL 10 -locate bank

We need to know where "bank" is inside the sequence.

In [11]:
bank_position = tokens_with_special.index("bank")

print(bank_position)

2


CELL 11 -Extract Bank Representation

Retrieve the vector for "bank". a contextualized word representation.

layer 1 , 0 senstence 0 , token

In [12]:
bank_vector = hidden_states[1][0, bank_position]

print(bank_vector.shape)

torch.Size([768])


CELL 12 - Compare layers


Observe that the representation changes.

In [13]:
for layer_num in [0, 1, 6, 12]:
    
    vec = hidden_states[layer_num][0, bank_position]

    print(
        f"Layer {layer_num}:",
        vec[:5]
    )

Layer 0: tensor([-0.2293, -0.6354, -1.2705, -1.1892, -0.3406])
Layer 1: tensor([-0.1287, -0.2373, -1.3220, -1.2427,  0.2241])
Layer 6: tensor([ 0.2771,  0.0125, -0.8137, -0.8707,  1.4581])
Layer 12: tensor([ 0.3164, -0.3892, -0.2163,  0.2654,  1.2507])


Cell 13 — Create a Reusable Function

In [14]:
def get_word_embedding(sentence, target_word):

    inputs = tokenizer(
        sentence,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.hidden_states

    tokens = tokenizer.convert_ids_to_tokens(
        inputs["input_ids"][0]
    )

    position = tokens.index(target_word)

    embeddings = []

    for layer in hidden_states:
        embeddings.append(
            layer[0, position].cpu().numpy()
        )

    return embeddings

Cell 14 — Test Function

In [17]:
embeddings = get_word_embedding(
    "The bank approved the loan.",
    "bank"
)

print(len(embeddings))
print(type(embeddings))

for i, vec in enumerate(embeddings):
    print(i, vec.shape)

13
<class 'list'>
0 (768,)
1 (768,)
2 (768,)
3 (768,)
4 (768,)
5 (768,)
6 (768,)
7 (768,)
8 (768,)
9 (768,)
10 (768,)
11 (768,)
12 (768,)


Cell 15 — Compare Contexts

In [16]:
s1 = "The bank approved the loan."
s2 = "The bank lowered interest rates."

emb1 = get_word_embedding(
    s1,
    "bank"
)

emb2 = get_word_embedding(
    s2,
    "bank"
)